In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

train_df = pd.read_csv("Imb_train_data.csv")
test_df  = pd.read_csv("Imb_test_data.csv")

tfidf = TfidfVectorizer(
    max_features=30000,     # more features for 20k reviews
    ngram_range=(1, 3),     # include tri-grams
    min_df=3,               # ignore words that appear in <3 docs
    max_df=0.8,             # ignore overly common words
    sublinear_tf=True,      # log-scaling term frequency
    stop_words='english'    # remove stopwords
)

X_train = tfidf.fit_transform(train_df['Review'])
X_test  = tfidf.transform(test_df['Review'])
y_train = train_df['Rating']
y_test  = test_df['Rating']


In [4]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

# Convert predictions to nearest integer (1–5)
y_pred_round = [round(y) for y in y_pred]
acc = accuracy_score(y_test, y_pred_round)

print("Linear Regression Accuracy:", acc)


Linear Regression Accuracy: 0.2446


In [6]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
y_pred = log_model.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Logistic Regression Accuracy: 0.4406

Classification Report:
               precision    recall  f1-score   support

           1       0.51      0.29      0.37       500
           2       0.35      0.20      0.26       750
           3       0.40      0.43      0.42      1250
           4       0.44      0.60      0.51      1500
           5       0.52      0.46      0.48      1000

    accuracy                           0.44      5000
   macro avg       0.44      0.40      0.41      5000
weighted avg       0.44      0.44      0.43      5000



In [7]:

from sklearn.svm import LinearSVC

svm_model = LinearSVC(C=10, class_weight='balanced')
svm_model.fit(X_train, y_train)
y_pred = svm_model.predict(X_test)

print("SVM Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


SVM Accuracy: 0.375

Classification Report:
               precision    recall  f1-score   support

           1       0.35      0.32      0.33       500
           2       0.27      0.27      0.27       750
           3       0.36      0.35      0.36      1250
           4       0.41      0.43      0.42      1500
           5       0.42      0.42      0.42      1000

    accuracy                           0.38      5000
   macro avg       0.36      0.36      0.36      5000
weighted avg       0.37      0.38      0.37      5000



In [8]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
y_pred = nb_model.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Naive Bayes Accuracy: 0.3702

Classification Report:
               precision    recall  f1-score   support

           1       1.00      0.02      0.03       500
           2       0.43      0.00      0.01       750
           3       0.33      0.31      0.32      1250
           4       0.36      0.88      0.51      1500
           5       0.74      0.13      0.22      1000

    accuracy                           0.37      5000
   macro avg       0.57      0.27      0.22      5000
weighted avg       0.50      0.37      0.28      5000



In [9]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(max_depth=100, random_state=42)
dt_model.fit(X_train, y_train)
y_pred = dt_model.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Decision Tree Accuracy: 0.3178

Classification Report:
               precision    recall  f1-score   support

           1       0.25      0.20      0.22       500
           2       0.20      0.15      0.17       750
           3       0.29      0.37      0.33      1250
           4       0.36      0.36      0.36      1500
           5       0.38      0.37      0.37      1000

    accuracy                           0.32      5000
   macro avg       0.30      0.29      0.29      5000
weighted avg       0.31      0.32      0.31      5000



In [10]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Load Imbalanced Dataset
im_train_df = pd.read_csv("Imb_train_data.csv")
im_test_df  = pd.read_csv("Imb_test_data.csv")


X_train, y_train = im_train_df["Review"], im_train_df["Rating"]
X_test, y_test   = im_test_df["Review"], im_test_df["Rating"]

# TF-IDF Vectorization
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words='english'
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

results = {}

print("🔧 Fine-Tuning on IMBALANCED Dataset\n")

# -----------------------------
# Logistic Regression
# -----------------------------
best_acc, best_params = 0, None
for C in [0.01, 0.1, 1, 10]:
    model = LogisticRegression(C=C, solver='liblinear', max_iter=2000)
    model.fit(X_train_tfidf, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_tfidf))
    if acc > best_acc:
        best_acc, best_params = acc, C
results["Logistic Regression"] = (best_acc, f"C={best_params}")
print(f"Logistic Regression {best_acc:.4f} | Best C={best_params}")

# -----------------------------
# Linear SVM
# -----------------------------
best_acc, best_params = 0, None
for C in [0.01, 0.1, 1, 10]:
    model = LinearSVC(C=C, class_weight='balanced', max_iter=5000)
    model.fit(X_train_tfidf, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_tfidf))
    if acc > best_acc:
        best_acc, best_params = acc, C
results["Linear SVM"] = (best_acc, f"C={best_params}")
print(f"Linear SVM {best_acc:.4f} | Best C={best_params}")

# -----------------------------
# Multinomial Naive Bayes
# -----------------------------
best_acc, best_params = 0, None
for alpha in [0.1, 0.5, 1.0, 2.0]:
    model = MultinomialNB(alpha=alpha)
    model.fit(X_train_tfidf, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_tfidf))
    if acc > best_acc:
        best_acc, best_params = acc, alpha
results["MultinomialNB"] = (best_acc, f"alpha={best_params}")
print(f"MultinomialNB {best_acc:.4f} | Best alpha={best_params}")

# -----------------------------
# Summary
# -----------------------------
print("\n📊 Best Model for IMBALANCED Dataset:")
best_model_name = max(results, key=lambda x: results[x][0])
print(f"{best_model_name}: Accuracy={results[best_model_name][0]:.4f} | Params={results[best_model_name][1]}")


🔧 Fine-Tuning on IMBALANCED Dataset

Logistic Regression 0.4324 | Best C=1
Linear SVM 0.4258 | Best C=0.1
MultinomialNB 0.4074 | Best alpha=0.1

📊 Best Model for IMBALANCED Dataset:
Logistic Regression: Accuracy=0.4324 | Params=C=1


In [11]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Load Imbalanced Dataset
im_train_df = pd.read_csv("Imb_train_data.csv")
im_test_df  = pd.read_csv("Imb_test_data.csv")


X_train, y_train = im_train_df["Review"], im_train_df["Rating"]
X_test, y_test   = im_test_df["Review"], im_test_df["Rating"]

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

results = {}

print("🔧 Fine-Tuning on IMBALANCED Dataset\n")

# -----------------------------
# Logistic Regression
# -----------------------------
print("Logistic Regression (varying C):")
for C in [0.01, 0.1, 1, 10]:
    model = LogisticRegression(C=C, solver='liblinear', max_iter=2000)
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    print(f"C={C} → Accuracy: {acc:.4f}")
    if acc > results.get("Logistic Regression", (0,))[0]:
        results["Logistic Regression"] = (acc, C)

# -----------------------------
# Linear SVM
# -----------------------------
print("\nLinear SVM (varying C):")
for C in [0.01, 0.1, 1, 10]:
    model = LinearSVC(C=C, class_weight='balanced', max_iter=5000)
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    print(f"C={C} → Accuracy: {acc:.4f}")
    if acc > results.get("Linear SVM", (0,))[0]:
        results["Linear SVM"] = (acc, C)

# -----------------------------
# Multinomial Naive Bayes
# -----------------------------
print("\nMultinomial Naive Bayes (varying alpha):")
for alpha in [0.1, 0.5, 1.0, 2.0]:
    model = MultinomialNB(alpha=alpha)
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    print(f"alpha={alpha} → Accuracy: {acc:.4f}")
    if acc > results.get("MultinomialNB", (0,))[0]:
        results["MultinomialNB"] = (acc, alpha)

# -----------------------------
# Summary of Best Models
# -----------------------------
print("\n📊 Best Model for IMBALANCED Dataset:")
best_model_name = max(results, key=lambda x: results[x][0])
best_acc, best_param = results[best_model_name]
print(f"{best_model_name}: Accuracy={best_acc:.4f} | Best Param={best_param}")


🔧 Fine-Tuning on IMBALANCED Dataset

Logistic Regression (varying C):
C=0.01 → Accuracy: 0.3006
C=0.1 → Accuracy: 0.3956
C=1 → Accuracy: 0.4324
C=10 → Accuracy: 0.4062

Linear SVM (varying C):
C=0.01 → Accuracy: 0.4234
C=0.1 → Accuracy: 0.4258
C=1 → Accuracy: 0.3952
C=10 → Accuracy: 0.3650

Multinomial Naive Bayes (varying alpha):
alpha=0.1 → Accuracy: 0.4074
alpha=0.5 → Accuracy: 0.4072
alpha=1.0 → Accuracy: 0.3992
alpha=2.0 → Accuracy: 0.3840

📊 Best Model for IMBALANCED Dataset:
Logistic Regression: Accuracy=0.4324 | Best Param=1


In [12]:
import pandas as pd
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from tabulate import tabulate  # for table view

# -----------------------------
# 1️⃣ Load imbalanced datasets
# -----------------------------
train_df = pd.read_csv("Imb_train_data.csv")
test_df  = pd.read_csv("Imb_test_data.csv")

X_train, y_train = train_df["Review"], train_df["Rating"]
X_test, y_test   = test_df["Review"], test_df["Rating"]

# -----------------------------
# 2️⃣ TF-IDF Vectorization (same as tuned for imbalanced data)
# -----------------------------
tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1,3),
    min_df=3,
    max_df=0.8,
    sublinear_tf=True,
    stop_words='english'
)
X_train_tfidf = tfidf.fit_transform(X_train)

# -----------------------------
# 3️⃣ Final Logistic Regression (C=1 for imbalanced data)
# -----------------------------
final_model = LogisticRegression(C=1, solver='liblinear', max_iter=2000)
final_model.fit(X_train_tfidf, y_train)

# -----------------------------
# 4️⃣ Select random test reviews (5 per rating)
# -----------------------------
ratings = [1, 2, 3, 4, 5]
sample_reviews = []

for r in ratings:
    subset = test_df[test_df['Rating'] == r]
    if len(subset) >= 5:
        review_samples = subset.sample(n=5, random_state=random.randint(0, 1000))
    else:
        review_samples = subset  # if fewer than 5 exist
    for _, row in review_samples.iterrows():
        sample_reviews.append((r, row['Review']))

# -----------------------------
# 5️⃣ Predict and store results
# -----------------------------
results = []

for idx, (true_rating, review_text) in enumerate(sample_reviews, start=1):
    X_vec = tfidf.transform([review_text])
    pred_rating = final_model.predict(X_vec)[0]
    results.append({
        "S.No": idx,
        "Actual Rating": true_rating,
        "Predicted Rating": pred_rating,
        "Review (First 200 chars)": review_text[:200] + ("..." if len(review_text) > 200 else "")
    })

# -----------------------------
# 6️⃣ Display as a formatted table
# -----------------------------
results_df = pd.DataFrame(results)

print("\n📊 Model Prediction Test on Random Reviews (Imbalanced Dataset):\n")
print(tabulate(results_df, headers="keys", tablefmt="grid", showindex=False))



📊 Model Prediction Test on Random Reviews (Imbalanced Dataset):

+--------+-----------------+--------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|   S.No |   Actual Rating |   Predicted Rating | Review (First 200 chars)                                                                                                                                                                                    |
+========+=================+====================+=============================================================================================================================================================================================================+
|      1 |               1 |                  1 | stella doro company sold baking believe north although recipe may machine used use cake mixer destroy texture called

In [13]:
import joblib

# Save the final trained model
joblib.dump(final_model, "Model_B.pkl")

['Model_B.pkl']

In [14]:
from google.colab import files

files.download("Model_B.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
joblib.dump(tfidf, "tfidf_imb.pkl")

['tfidf_imb.pkl']

In [16]:
files.download("tfidf_imb.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>